# v37 — повысит ли TransReID качество рабочего v25?
Run All: 9 систем на calibration → один зафиксированный победитель → сравнение с v25 на исходной validation.
**Без нового обучения, подбора порога и автоматической замены MVP.** Используются сохранённые T12/T22 @1800.
Новые доли: 5/10/15/20%; реранкер 20/3/0.50 и candidate-ветвь v25 неизменны.
Обе выборки уже использовались: это development, не независимый тест. Ограничения времени нет.

In [1]:
import os
os.environ['ORT_DISABLE_TELEMETRY'] = '1'  # до импорта runtime, защита от прежнего SIGABRT
from pathlib import Path
import sys, subprocess, time

DEVICE = 'mps'
RUN_NAME = 'system_v1'  # то же имя = проверенное продолжение

ROOT = next((p for root in [Path.cwd(), *Path.cwd().parents]
             for p in (root, root / 'Car-classification-MSK')
             if (p / 'training/transreid_system.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Открой notebook внутри Car-classification-MSK')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
STARTED = time.perf_counter()
print('Python:', sys.executable)
print('Project:', ROOT)
print('Device:', DEVICE, '| Run:', RUN_NAME, '| training: OFF | time limit: NONE')


Python: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python
Project: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK
Device: mps | Run: system_v1 | training: OFF | time limit: NONE


In [2]:
print('Этап 1/4: проверки компонентов')
subprocess.run([sys.executable, '-m', 'pytest', '-q',
    'tests/test_transreid_system.py', 'tests/test_dual_role_inference.py',
    'tests/test_retrieval_policy.py'], check=True, cwd=ROOT,
    env={**os.environ, 'ORT_DISABLE_TELEMETRY': '1'})
from training import transreid_system as experiment


Этап 1/4: проверки компонентов
........................................................................ [ 92%]
......                                                                   [100%]
78 passed in 4.85s


In [3]:
print('Этап 2/4: проверка исходных файлов и фиксация протокола; хеширование может занять время')
context = experiment.prepare(run_name=RUN_NAME, device=DEVICE)
print('Output:', context['output'])
print('Signature:', context['signature'])
print('Frozen threshold:', context['manifest']['threshold'])
for name, spec in context['manifest']['systems'].items(): print(name, spec)


Этап 2/4: проверка исходных файлов и фиксация протокола; хеширование может занять время
PREFLIGHT OK: 9 systems, 740 trained IDs, device=mps; original validation stays closed until selection
Output: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_37_transreid_system/runs/system_v1
Signature: 90c0f606afca5857fc8f37cbd9b1fefeb5613237ddaa8e88f23ed0050369ee76
Frozen threshold: 0.534365177154541
V25_control {'member': None, 'weight': 0.0}
T12_w05 {'member': 'T12', 'weight': 0.05}
T22_w05 {'member': 'T22', 'weight': 0.05}
T12_w10 {'member': 'T12', 'weight': 0.1}
T22_w10 {'member': 'T22', 'weight': 0.1}
T12_w15 {'member': 'T12', 'weight': 0.15}
T22_w15 {'member': 'T22', 'weight': 0.15}
T12_w20 {'member': 'T12', 'weight': 0.2}
T22_w20 {'member': 'T22', 'weight': 0.2}


In [4]:
print('Этап 3/4: новые признаки → calibration → фиксация → validation → экспорт')
results = experiment.run(context, allow_outer=True)
assert results['status'] == 'complete' and results['candidate_unchanged']
assert not results['promoted'] and not results['threshold_fit']
assert results['optimizer_updates'] == results['bn_updates'] == 0


Этап 3/4: новые признаки → calibration → фиксация → validation → экспорт
CHECK: 13223 protected files and frozen source
STAGE runtime_probe: start
Dual-role CPU: 8/8 images
using stride: 16, and patch number is num_y16 * num_x16
using drop_out rate is : 0.0
using attn_drop_out rate is : 0.0
using drop_path rate is : 0.1
PROBE T12, batch=1: max error=0
PROBE T12, batch=8: max error=0
PROBE T12, batch=16: max error=0
PROBE T12, batch=32: max error=0
using stride: 16, and patch number is num_y16 * num_x16
using drop_out rate is : 0.0
using attn_drop_out rate is : 0.0
using drop_path rate is : 0.1
PROBE T22, batch=1: max error=2.98e-08
PROBE T22, batch=8: max error=0
PROBE T22, batch=16: max error=0
PROBE T22, batch=32: max error=0
STAGE runtime_probe: complete in 7.0s
STAGE features_calibration_T12: start
using stride: 16, and patch number is num_y16 * num_x16
using drop_out rate is : 0.0
using attn_drop_out rate is : 0.0
using drop_path rate is : 0.1
FEATURES 16/1202
FEATURES 144/1202
FE

In [5]:
from IPython.display import Markdown, display
print('Этап 4/4: итоговый отчёт')
display(Markdown((context['output'] / 'REPORT.md').read_text(encoding='utf-8')))
print('Выбрано на calibration:', results['selection']['selected'])
for name, report in results['evaluations']['validation'].items():
    print(name, '| validation mAP@10:', report['ranking']['mAP@10'])
print(f'Текущий запуск notebook: {(time.perf_counter()-STARTED)/60:.1f} мин; не benchmark')
print('Отчёт:', context['output'] / 'REPORT.md')
print('Готово. Рабочий v25 не изменён. Можно прислать: подсчёты завершились.')


Этап 4/4: итоговый отчёт


# v37 — v25 + TransReID без нового обучения

Выбор на calibration: **T12_w10**.
MVP_fusion_v25 не изменён. Порог, кандидат, confidence и отказы остаются от v25.

| Split | Система | mAP@10 | Δ к v25, п.п. | Rank-1 | F1 | TNR |
|---|---|---:|---:|---:|---:|---:|
| calibration | V25_control | 0.826410 | +0.0000 | 0.841463 | 0.813483 | 0.704918 |
| calibration | T12_w05 | 0.832267 | +0.5857 | 0.845528 | 0.813483 | 0.704918 |
| calibration | T22_w05 | 0.829774 | +0.3365 | 0.841463 | 0.813483 | 0.704918 |
| calibration | T12_w10 | 0.834048 | +0.7638 | 0.845528 | 0.813483 | 0.704918 |
| calibration | T22_w10 | 0.832488 | +0.6078 | 0.841463 | 0.813483 | 0.704918 |
| calibration | T12_w15 | 0.827163 | +0.0753 | 0.837398 | 0.813483 | 0.704918 |
| calibration | T22_w15 | 0.830275 | +0.3865 | 0.845528 | 0.813483 | 0.704918 |
| calibration | T12_w20 | 0.822595 | -0.3815 | 0.833333 | 0.813483 | 0.704918 |
| calibration | T22_w20 | 0.824092 | -0.2318 | 0.833333 | 0.813483 | 0.704918 |
| validation | V25_control | 0.828957 | +0.0000 | 0.829960 | 0.795455 | 0.709677 |
| validation | T12_w10 | 0.821831 | -0.7127 | 0.809717 | 0.795455 | 0.709677 |

Validation AP: улучшилось 14, ухудшилось 26, без изменения 207.
Top-10 изменились у 281 запросов; top-1 исправлен у 1, испорчен у 6.

## Как читать результат

Подбиралась только доля новой модели: 5/10/15/20%, отдельно T12 и T22, плюс контроль с нулевой долей.
Косинус до графа: (1-w) × v25 + w × TransReID; внутри v25 равные доли MVP и R1 equal3.
Реранкер legacy 20/3/0.50 зафиксирован, не подбирался. Точные равенства решаются в пользу контроля, меньшей доли, затем T12.
До чтения validation-признаков/оценок записан frozen_selection.json; validation считает только контроль и выбранную систему.
Исходная validation уже использовалась раньше: это development-сравнение, не независимый тест и не обещание hidden-test результата.
T12/T22 — конкретные сохранённые checkpoints на 740 train-ID; новый full-train refit не выполнялся. v25 обучался на своём полном train.
Всего 0 optimizer updates, 0 BN updates; исходные bbox, разбиения и данные не менялись, NiVe не используется.
CSV каждого экспорта содержит top-10 даже при отказе; candidates.csv совпадает с v25 побайтно.
embeddings.npy содержит реальные unit-блоки: MVP512 + R1_1536 [+ T12_384 или T22_1920]. Это не единый ranking cosine-vector.
NPY replay использует явные slices/веса. Экспорты лежат в tasks/export_<split>_<system>/export/.
Новые признаки — PyTorch float32 на явно выбранном устройстве; v25 — проверенный прежний CPU ONNX-кэш.
Кэш не выдаётся за новый замер скорости. Это исследовательский runner, не готовый новый ONNX-профиль продукта.
Независимость изображений: свежие batch 1/8/16/32, перестановка/одиночное изображение; буферы модели неизменны.
CPU/GPU скорость новой полной системы, Linux и экспорт в продукт проверяются отдельно только при полезном качестве.
Один новый encoder добавляется к четырём encoder v25. Ограничения junk/top-10 и остаточного сигнала номеров не устранены.
Ни результаты, ни победа на validation не переключают MVP автоматически. Новые пороги/веса по validation не подбираются.


Выбрано на calibration: T12_w10
V25_control | validation mAP@10: 0.8289573904235558
T12_w10 | validation mAP@10: 0.8218308745000474
Текущий запуск notebook: 2.1 мин; не benchmark
Отчёт: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_37_transreid_system/runs/system_v1/REPORT.md
Готово. Рабочий v25 не изменён. Можно прислать: подсчёты завершились.
